# 42173 Advanced Natural Language Processing
## Evaluation: standard RAG vs CRAG on PubHealth (Qwen2.5-3B generator)

This notebook compares **standard RAG** with **CRAG**, with the LLM on its own (no retrieval) as the baseline, on the
same 200 PubHealth health claims. The generator (`src/crag/generator.py`, Qwen2.5-Instruct), prompt, claims and
decoding are the same in every setting, so any difference in accuracy comes from the knowledge the generator is given.

| Setting | Knowledge given to the generator |
|---|---|
| No retrieval | none, the model answers from what it learnt in pre-training |
| Standard RAG | the top-k MedRAG Textbooks passages from Contriever |
| CRAG | the same passages after the retrieval evaluator, action trigger, knowledge refinement and web search |

**What is evaluated**

1. **Main comparison** (section 7): accuracy with 95% bootstrap intervals, macro F1, balanced accuracy, AUROC of the
   generator's P(true), confusion matrices, exact McNemar tests, accuracy by CRAG action, and time per claim.
2. **Ablations** (section 8), following Tables 2, 3 and 11 of the CRAG paper (Yan et al., 2024): CRAG without
   refinement, without query rewriting and without web search; one action forced for every claim; and an oracle
   context (the fact-checkers' own explanation) as an upper bound.
3. **Threshold sensitivity** (section 9): CRAG accuracy for every pair of action thresholds, computed exactly from the
   forced-action runs, since `UPPER` and `LOWER` in `src/crag/actions.py` are still placeholders.
4. **Quality of the knowledge** (section 10): a sentence-level evidence table scored by the retrieval evaluator, how
   close each method's knowledge is to the fact-checkers' explanation, and whether the evaluator separates a claim's
   gold explanation from a wrong one.

The group's code is used unchanged: the `MODELS` table and index path from `scripts/evaluate.py`, `load_pubhealth`,
`run_rag`, `correct_knowledge`, `refine`, `rewrite_query`, `web_search` and `predict_label`. Results are saved after
every claim, so a run that stops part-way resumes where it stopped.

## 1. Setup

**Where to run it.** Open this notebook from the `notebooks/` folder of the repo, so it can find `src/crag`.
On SageMaker or a local machine, upload or unzip the repo (with `models/` and `indexes/`) and run all cells.
On Google Colab, uncomment the clone lines in the next cell first. The next cell installs the requirements only if
something is missing.

In [ ]:
# On Google Colab, uncomment these lines to get the repo (with its Git LFS model files).
# !git lfs install
# !git clone https://github.com/hulemihi/42173-ANLP-CRAG.git
# %cd 42173-ANLP-CRAG/notebooks

import importlib.util
import subprocess
import sys

missing = [m for m in ("datasets", "transformers", "sentencepiece", "dotenv", "rank_bm25", "matplotlib")
           if importlib.util.find_spec(m) is None]
if missing:
    print("Installing the requirements, missing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)
else:
    print("Requirements already installed")

In [ ]:
import importlib.util
import json
import math
import os
import re
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# The repo root is the folder that contains src/crag (this notebook lives in notebooks/)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "crag").is_dir())
sys.path.insert(0, str(ROOT / "src"))

from crag import actions
from crag import evaluator as crag_evaluator
from crag.data import load_pubhealth, load_textbooks_corpus
from crag.evaluator import evaluate_relevance, load_evaluator
from crag.generator import load_generator, predict_label
from crag.pipeline import check_crag_ready, correct_knowledge, run_rag
from crag.refinement import refine, split_into_strips
from crag.retriever import _encode, build_index, retrieve
from crag.web_search import rewrite_query, web_search

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Repo root:", ROOT)
print("Using device:", DEVICE, f"({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else "")

In [ ]:
# Reuse the model table and index path from scripts/evaluate.py, so this notebook and the script stay in sync
spec = importlib.util.spec_from_file_location("evaluate_script", ROOT / "scripts" / "evaluate.py")
evaluate_script = importlib.util.module_from_spec(spec)
spec.loader.exec_module(evaluate_script)

MODELS = evaluate_script.MODELS
INDEX_PATH = evaluate_script.INDEX_PATH
MODELS

## 2. Settings

`MODEL_KEY` picks the generator from `MODELS`. The 3B model is the default, as in `scripts/evaluate.py`.

- **GPU** (recommended). bf16 on GPUs that support it (A10G on `ml.g5`, A100, L4). On older GPUs without bf16, such as
  the T4 on `ml.g4dn` or Colab's free GPU, the weights run in float32 instead (about 13 GB of GPU memory for 3B).
- **CPU.** It still runs in float32, but slowly: on a 16-thread laptop CPU the main comparison takes about 25 seconds
  per claim and the ablations about the same again, so all 200 claims take about four hours.

Every part saves its results after each claim and resumes where it stopped if the notebook is run again.

In [ ]:
MODEL_KEY = "3b"          # "0.5b", "3b" or "7b" (7b is loaded in 4-bit and needs a GPU)
N_CLAIMS = 200            # same fixed, stratified PubHealth subset as scripts/evaluate.py
K = 5                     # passages retrieved per claim
RUN_ABLATIONS = True      # section 6: CRAG without each component, forced actions and the oracle context
N_EXAMPLES = 3            # claims shown with a full RAG and CRAG response at the end
FLOAT32_WITHOUT_BF16 = True  # on a CPU, or a GPU without bf16 (e.g. T4), run the bf16 weights in float32: much
                             # faster there, but twice the memory (about 13 GB for 3B). Ignored for the 4-bit 7B.
RESUME = True             # skip claims that are already in the results files
SEED = 42

In [ ]:
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)
TAG = f"pubhealth_n{N_CLAIMS}_{MODEL_KEY}"
RESULTS_PATH = RESULTS_DIR / f"crag_{TAG}.csv"                 # main comparison, one row per claim
ABLATION_PATH = RESULTS_DIR / f"crag_ablations_{TAG}.csv"      # ablations, one row per claim
KNOWLEDGE_PATH = RESULTS_DIR / f"crag_knowledge_{TAG}.jsonl"   # passages, strips, queries and web results
BACKOFF_PATH = RESULTS_DIR / f"crag_backoff_{TAG}.csv"         # the back-off search variant, one row per claim

print("Generator:", MODELS[MODEL_KEY])
print("Results files:", ", ".join(p.name for p in (RESULTS_PATH, ABLATION_PATH, KNOWLEDGE_PATH, BACKOFF_PATH)))
print("Web search:", "Tavily + PubMed" if os.getenv("TAVILY_API_KEY") else "PubMed only (no TAVILY_API_KEY in .env)")

## 3. Load the PubHealth claims

In [ ]:
claims = load_pubhealth(n=N_CLAIMS)
claims.insert(0, "id", range(len(claims)))

majority = claims.label.value_counts(normalize=True).max()
print(f"{len(claims)} claims: {claims.label.value_counts().to_dict()}")
print(f"Always answering the majority label gives {majority:.1%} accuracy")
claims[["id", "claim", "label"]].head()

## 4. Load the retriever index, the generator and the retrieval evaluator

In [ ]:
t0 = time.time()
index = build_index(load_textbooks_corpus(), cache_path=INDEX_PATH)
print(f"Contriever index: {len(index.passages):,} passages, ready in {time.time() - t0:.0f}s")

t0 = time.time()
model, tokenizer = load_generator(**MODELS[MODEL_KEY])
has_bf16 = DEVICE == "cuda" and torch.cuda.get_device_capability()[0] >= 8   # Ampere or newer
if FLOAT32_WITHOUT_BF16 and not has_bf16 and not MODELS[MODEL_KEY].get("load_in_4bit"):
    model = model.float()
n_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Generator: {MODELS[MODEL_KEY]['model_name']} ({n_params:.2f}B parameters, "
      f"{next(model.parameters()).dtype}), ready in {time.time() - t0:.0f}s")

eval_model, eval_tokenizer = load_evaluator()
problems = check_crag_ready()
print("CRAG stages ready" if not problems else "CRAG problems:\n" + "\n".join(problems))
print(f"Action thresholds: UPPER = {actions.UPPER}, LOWER = {actions.LOWER}")

## 5. Main comparison: no retrieval, standard RAG and CRAG

For each claim, the label is predicted three times with `explain=False`, exactly as in `scripts/evaluate.py`
(one forward pass, the label with the highest next-token probability):

1. **No retrieval**: `predict_label` with no passages.
2. **Standard RAG**: `run_rag`, the top-k textbook passages.
3. **CRAG**: the same steps as `pipeline.run_crag` (retrieve, `correct_knowledge`, `predict_label`), written out here
   so the evaluator scores and the size of the corrected knowledge can be saved as well.

In [ ]:
WEB_PREFIXES = ("[PubMed]", "[Web]")
LABEL_COLUMNS = ["gold", "no_retrieval", "rag", "crag"]


def as_labels(frame, columns):
    """pandas reads the words true / false back from a CSV as booleans, so turn every label back into the word."""
    for c in columns:
        if c in frame.columns:
            frame[c] = frame[c].map(lambda v: str(v).strip().lower())
    return frame


def evaluate_claim(claim):
    row = {}

    t = time.time()
    none = predict_label(model, tokenizer, claim, [], explain=False)
    row.update(no_retrieval=none.label, no_retrieval_p_true=none.probs["true"],
               seconds_no_retrieval=time.time() - t)

    t = time.time()
    rag = run_rag(claim, index, model, tokenizer, k=K, explain=False)
    row.update(rag=rag.label, rag_p_true=rag.probs["true"], seconds_rag=time.time() - t)

    t = time.time()
    passages = retrieve(index, claim, k=K)
    knowledge, action, scores = correct_knowledge(claim, passages, eval_model, eval_tokenizer,
                                                  gen_model=model, gen_tokenizer=tokenizer)
    crag = predict_label(model, tokenizer, claim, knowledge, explain=False)
    row.update(crag=crag.label, crag_p_true=crag.probs["true"], crag_action=action,
               crag_scores=json.dumps([round(float(s), 4) for s in scores]),
               crag_max_score=max(scores),
               crag_knowledge_items=len(knowledge),
               crag_web_items=sum(k.startswith(WEB_PREFIXES) for k in knowledge),
               crag_knowledge_words=sum(len(k.split()) for k in knowledge),
               seconds_crag=time.time() - t)
    return row

In [ ]:
rows = []
if RESUME and RESULTS_PATH.exists():
    saved = as_labels(pd.read_csv(RESULTS_PATH), LABEL_COLUMNS)
    if "id" in saved.columns:
        rows = saved[saved.id.isin(claims.id)].to_dict("records")
done = {r["id"] for r in rows}
print(f"Resuming with {len(done)} claims already done" if done else "Starting a new run")

start = time.time()
for r in tqdm(claims.itertuples(), total=len(claims), desc="Main comparison"):
    if r.id in done:
        continue
    rows.append({"id": r.id, "claim": r.claim, "gold": r.label, **evaluate_claim(r.claim)})
    pd.DataFrame(rows).sort_values("id").to_csv(RESULTS_PATH, index=False)

results = as_labels(pd.DataFrame(rows).sort_values("id").reset_index(drop=True), LABEL_COLUMNS)
results.to_csv(RESULTS_PATH, index=False)
print(f"{len(results)} claims in {RESULTS_PATH.name} ({time.time() - start:.0f}s in this session)")
results.head()

## 6. Ablations

Each variant changes one part of CRAG and keeps everything else (claims, passages, evaluator scores, generator) the
same. They follow the ablations in the CRAG paper (Tables 2, 3 and 11):

| Variant | Knowledge given to the generator |
|---|---|
| CRAG without refinement | whole passages instead of the refined strips |
| CRAG without query rewriting | web search with the claim itself instead of the rewritten keywords |
| CRAG without web search | refined strips only (Incorrect therefore gets no knowledge) |
| CRAG with back-off search | notebook-only variant: if PubMed finds nothing for the rewritten keywords, search again with fewer of them |
| Correct for every claim | refined strips, whatever the evaluator says |
| Incorrect for every claim | web search results only |
| Ambiguous for every claim | refined strips plus web search results |
| Oracle context | the PubHealth fact-checkers' explanation for the claim (an upper bound, not a real system) |

The rewritten query is usually a comma-separated list of 4 to 8 phrases, and PubMed returns only articles that
match every term, so it often finds nothing. The back-off variant is not part of the group's code. It shows what the
Incorrect and Ambiguous actions can do when the web search does return evidence.

A forward pass is only run when a variant's knowledge differs from knowledge already scored for that claim, so
variants that end up with the same knowledge as RAG, CRAG or no retrieval reuse those predictions. The forced-action
variants also make the threshold analysis in section 9 exact.

In [ ]:
ABLATIONS = {
    "crag_no_refine": "CRAG w/o refinement",
    "crag_no_rewrite": "CRAG w/o query rewriting",
    "crag_no_web": "CRAG w/o web search",
    "all_correct": "Correct for every claim",
    "all_incorrect": "Incorrect for every claim",
    "all_ambiguous": "Ambiguous for every claim",
    "oracle": "Oracle context",
}
C, I, A = actions.CORRECT, actions.INCORRECT, actions.AMBIGUOUS


def score_fn(question, text):
    return evaluate_relevance(eval_model, eval_tokenizer, question, text)


def ablate_claim(row, explanation):
    claim = row["claim"]
    t = time.time()
    passages = retrieve(index, claim, k=K)
    scores = [score_fn(claim, p) for p in passages]
    action = actions.decide_action(scores)
    refined = refine(claim, passages, score_fn)
    query = rewrite_query(claim, model=model, tokenizer=tokenizer)
    web = web_search(query)
    web_raw = web_search(claim)
    crag_knowledge = {C: refined, I: web, A: refined + web}[action]

    variants = {
        "crag_no_refine": {C: passages, I: web, A: passages + web}[action],
        "crag_no_rewrite": {C: refined, I: web_raw, A: refined + web_raw}[action],
        "crag_no_web": {C: refined, I: [], A: refined}[action],
        "all_correct": refined,
        "all_incorrect": web,
        "all_ambiguous": refined + web,
        "oracle": [explanation],
    }
    # Predictions already made in the main comparison for the same knowledge
    known = {(): (row["no_retrieval"], row["no_retrieval_p_true"]),
             tuple(passages): (row["rag"], row["rag_p_true"]),
             tuple(crag_knowledge): (row["crag"], row["crag_p_true"])}
    out = {"id": row["id"], "action_check": action, "query": query, "refined_strips": len(refined),
           "web_items": len(web), "raw_web_items": len(web_raw)}
    passes = 0
    for name, knowledge in variants.items():
        key = tuple(k for k in knowledge if k and k.strip())
        if key not in known:
            pred = predict_label(model, tokenizer, claim, list(key), explain=False)
            known[key] = (pred.label, pred.probs["true"])
            passes += 1
        out[name], out[f"{name}_p_true"] = known[key]
    out.update(forward_passes=passes, seconds_ablation=time.time() - t)

    record = {"id": row["id"], "claim": claim, "gold": row["gold"], "action": action,
              "scores": [round(float(s), 4) for s in scores], "passages": passages, "refined": refined,
              "query": query, "web": web, "web_raw": web_raw}
    return out, record

In [ ]:
explanations = dict(zip(claims.id, claims.explanation))
abl_rows = []
if RESUME and ABLATION_PATH.exists() and KNOWLEDGE_PATH.exists():
    abl_rows = as_labels(pd.read_csv(ABLATION_PATH), list(ABLATIONS)).to_dict("records")
    kept = {r["id"] for r in abl_rows}
    # keep the knowledge file in step with the ablation file
    records = [json.loads(l) for l in KNOWLEDGE_PATH.read_text().splitlines() if l.strip()]
    KNOWLEDGE_PATH.write_text("".join(json.dumps(r) + "\n" for r in records if r["id"] in kept))
elif KNOWLEDGE_PATH.exists():
    KNOWLEDGE_PATH.unlink()
done = {r["id"] for r in abl_rows}

if RUN_ABLATIONS:
    print(f"Resuming with {len(done)} claims already done" if done else "Starting the ablations")
    start = time.time()
    for row in tqdm(results.to_dict("records"), desc="Ablations"):
        if row["id"] in done:
            continue
        out, record = ablate_claim(row, explanations[row["id"]])
        abl_rows.append(out)
        with open(KNOWLEDGE_PATH, "a") as f:
            f.write(json.dumps(record) + "\n")
        pd.DataFrame(abl_rows).sort_values("id").to_csv(ABLATION_PATH, index=False)
    print(f"{len(abl_rows)} claims in {ABLATION_PATH.name} ({time.time() - start:.0f}s in this session)")

ablations = as_labels(pd.DataFrame(abl_rows).sort_values("id").reset_index(drop=True), list(ABLATIONS)) \
    if abl_rows else None
if ablations is not None:
    ablations.to_csv(ABLATION_PATH, index=False)
knowledge = {r["id"]: r for r in (json.loads(l) for l in KNOWLEDGE_PATH.read_text().splitlines() if l.strip())} \
    if KNOWLEDGE_PATH.exists() else {}
if ablations is not None:
    same = (ablations.set_index("id").action_check == results.set_index("id").crag_action).mean()
    print(f"CRAG action reproduced for {same:.1%} of claims; {ablations.forward_passes.sum()} extra forward passes")

### Back-off search (notebook-only variant)

The rewritten query is a list of keywords, and PubMed only returns articles that match every one of them, so CRAG's
web search often finds nothing. This step keeps everything else from CRAG (the action, the refined strips, the
rewritten query) and, when PubMed returns nothing, searches again with only the first 4, 3 or 2 keywords. It is not
part of the group's code; it shows what the Incorrect and Ambiguous actions can do when the search does return
evidence. It reuses the saved queries, so only this step needs the generator again.

In [ ]:
items = list(ABLATIONS.items())
ABLATIONS = dict(items[:3] + [("crag_backoff", "CRAG with back-off search")] + items[3:])
BACKOFF_COLUMNS = ["crag_backoff", "crag_backoff_p_true", "backoff_web_items", "query_backoff"]


def crag_knowledge_of(rec):
    return {C: rec["refined"], I: rec["web"], A: rec["refined"] + rec["web"]}[rec["action"]]


def backoff_search(query):
    """web_search with the rewritten keywords; if nothing is found, retry with the first 4, 3, then 2 of them."""
    parts = [p.strip() for p in re.split(r"[,;\n]+", query) if p.strip()]
    if len(parts) < 2:
        parts = query.split()
    tried = []
    for n in [len(parts), 4, 3, 2]:
        q = " ".join(parts[:n])
        if n > len(parts) or q in tried:
            continue
        tried.append(q)
        found = web_search(q)
        if found:
            return found, q
    return [], tried[-1] if tried else query


def backoff_claim(rec, row, abl_row):
    if rec["action"] == C:  # Correct never searches the web, so the variant equals CRAG
        return {"id": rec["id"], "crag_backoff": row["crag"], "crag_backoff_p_true": row["crag_p_true"],
                "backoff_web_items": 0, "query_backoff": ""}
    web_b, query_b = backoff_search(rec["query"])
    knowledge_b = {I: web_b, A: rec["refined"] + web_b}[rec["action"]]
    known = {(): (row["no_retrieval"], row["no_retrieval_p_true"]),
             tuple(crag_knowledge_of(rec)): (row["crag"], row["crag_p_true"]),
             tuple(rec["refined"]): (abl_row["all_correct"], abl_row["all_correct_p_true"])}
    key = tuple(k for k in knowledge_b if k and k.strip())
    if key in known:
        label, p_true = known[key]
    else:
        pred = predict_label(model, tokenizer, row["claim"], list(key), explain=False)
        label, p_true = pred.label, pred.probs["true"]
    return {"id": rec["id"], "crag_backoff": label, "crag_backoff_p_true": p_true,
            "backoff_web_items": len(web_b), "query_backoff": query_b}


bo_rows = []
if RESUME and BACKOFF_PATH.exists():
    bo_rows = as_labels(pd.read_csv(BACKOFF_PATH), ["crag_backoff"]).to_dict("records")
done = {r["id"] for r in bo_rows}
if ablations is not None:
    rows_by_id = results.set_index("id", drop=False).to_dict("index")
    abl_by_id = ablations.set_index("id", drop=False).to_dict("index")
    for rec in tqdm(sorted(knowledge.values(), key=lambda r: r["id"]), desc="Back-off search"):
        if rec["id"] in done or rec["id"] not in rows_by_id:
            continue
        bo_rows.append(backoff_claim(rec, rows_by_id[rec["id"]], abl_by_id[rec["id"]]))
        pd.DataFrame(bo_rows).sort_values("id").to_csv(BACKOFF_PATH, index=False)
    backoff = as_labels(pd.DataFrame(bo_rows), ["crag_backoff"])
    ablations = ablations.drop(columns=[c for c in BACKOFF_COLUMNS if c in ablations.columns]).merge(backoff, on="id")
    searched = ablations.action_check != C
    print(f"Of the {searched.sum()} claims where CRAG searches the web, PubMed found results for "
          f"{int((ablations.web_items[searched] > 0).sum())} with the rewritten query and "
          f"{int((ablations.backoff_web_items[searched] > 0).sum())} with the back-off search")

## 7. Main comparison: results

In [ ]:
SETTINGS = {"no_retrieval": "No retrieval", "rag": "Standard RAG", "crag": "CRAG"}
LABELS = ("true", "false")
COLOURS = {"no_retrieval": "#64748B", "rag": "#D97706", "crag": "#4F46E5"}
_boot = {}


def boot(n):
    """2,000 bootstrap resamples of n claims, the same for every setting so differences are paired."""
    if n not in _boot:
        _boot[n] = np.random.default_rng(SEED).integers(0, n, size=(2000, n))
    return _boot[n]


def macro_f1(gold, pred):
    f1s = []
    for label in LABELS:
        tp = ((pred == label) & (gold == label)).sum()
        fp = ((pred == label) & (gold != label)).sum()
        fn = ((pred != label) & (gold == label)).sum()
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * precision * recall / (precision + recall) if precision + recall else 0.0)
    return float(np.mean(f1s))


def balanced_accuracy(gold, pred):
    return float(np.mean([(pred[gold == label] == label).mean() for label in LABELS]))


def auroc(gold, p_true):
    """Probability that a true claim gets a higher P(true) than a false claim (ties count half)."""
    ranks = pd.Series(p_true).rank().to_numpy()
    pos = (np.asarray(gold) == "true")
    n_pos, n_neg = pos.sum(), (~pos).sum()
    return float((ranks[pos].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def ci(correct):
    """95% bootstrap interval of the accuracy."""
    correct = np.asarray(correct, dtype=float)
    means = correct[boot(len(correct))].mean(axis=1)
    return np.percentile(means, [2.5, 97.5])


def evaluate(frame, cols):
    out = []
    for col, name in cols.items():
        correct = (frame[col] == frame.gold).to_numpy()
        lo, hi = ci(correct)
        out.append({"setting": name, "accuracy": correct.mean(), "ci_low": lo, "ci_high": hi,
                    "macro_f1": macro_f1(frame.gold, frame[col]),
                    "balanced_accuracy": balanced_accuracy(frame.gold, frame[col]),
                    "auroc_p_true": auroc(frame.gold, frame[f"{col}_p_true"]),
                    "predicted_true": int((frame[col] == "true").sum()),
                    "mean_p_true": frame[f"{col}_p_true"].mean()})
    return pd.DataFrame(out).set_index("setting")


for col in SETTINGS:
    results[f"{col}_correct"] = results[col] == results.gold
summary = evaluate(results, SETTINGS)
summary["seconds_per_claim"] = [results[f"seconds_{c}"].mean() for c in SETTINGS]

always_true = (results.gold == "true").mean()
print(f"{MODELS[MODEL_KEY]['model_name']} on {len(results)} PubHealth claims "
      f"({results.gold.value_counts().to_dict()}). Always answering 'true' gives {always_true:.1%} accuracy, "
      f"0.5 balanced accuracy and 0.5 AUROC.")
summary.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
ax.bar(summary.index, summary.accuracy, color=list(COLOURS.values()),
       yerr=[summary.accuracy - summary.ci_low, summary.ci_high - summary.accuracy], capsize=6)
ax.axhline(always_true, ls="--", color="black", lw=1, label=f"always 'true' ({always_true:.1%})")
for i, v in enumerate(summary.accuracy):
    ax.text(i + 0.05, v + 0.01, f"{v:.1%}", ha="left")
ax.set_ylim(0, 1)
ax.set_ylabel("Accuracy (95% bootstrap interval)")
ax.set_title(f"PubHealth accuracy, Qwen2.5-{MODEL_KEY.upper()} (n={len(results)})")
ax.legend(loc="upper right")

bins = np.linspace(0, 1, 21)
for col, name in SETTINGS.items():
    axes[1].hist(results[f"{col}_p_true"], bins=bins, alpha=0.55, color=COLOURS[col], label=name)
axes[1].axvline(0.5, ls="--", color="black", lw=1)
axes[1].set_xlabel("P(true) from the generator")
axes[1].set_ylabel("Claims")
axes[1].set_title("How confident the generator is that a claim is true")
axes[1].legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / f"fig_accuracy_{TAG}.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Confusion matrices (rows = gold label, columns = predicted label)
for col, name in SETTINGS.items():
    print(name)
    print(pd.crosstab(results.gold, results[col]).reindex(index=LABELS, columns=LABELS, fill_value=0), "\n")

### Paired comparison (exact McNemar test)

All settings answer the same claims, so they are compared on the claims where they disagree. A p-value below 0.05
means the difference is unlikely to be chance. The bootstrap interval is for the difference in accuracy.

In [ ]:
def mcnemar(a_correct, b_correct):
    """Exact two-sided McNemar test. Returns (only A right, only B right, p-value)."""
    a_correct, b_correct = np.asarray(a_correct, bool), np.asarray(b_correct, bool)
    only_a = int((a_correct & ~b_correct).sum())
    only_b = int((~a_correct & b_correct).sum())
    n = only_a + only_b
    if n == 0:
        return only_a, only_b, 1.0
    tail = sum(math.comb(n, i) for i in range(min(only_a, only_b) + 1)) / 2 ** n
    return only_a, only_b, min(1.0, 2 * tail)


def compare(frame, a, b, names):
    a_ok = (frame[a] == frame.gold).to_numpy()
    b_ok = (frame[b] == frame.gold).to_numpy()
    only_a, only_b, p = mcnemar(a_ok, b_ok)
    idx = boot(len(a_ok))
    diff = (a_ok.astype(float)[idx] - b_ok.astype(float)[idx]).mean(axis=1)
    lo, hi = np.percentile(diff, [2.5, 97.5])
    return {"comparison": f"{names[a]} vs {names[b]}", "difference": a_ok.mean() - b_ok.mean(),
            "ci_low": lo, "ci_high": hi, "only first right": only_a, "only second right": only_b, "p_value": p}


paired = pd.DataFrame([compare(results, a, b, SETTINGS)
                       for a, b in [("crag", "rag"), ("rag", "no_retrieval"), ("crag", "no_retrieval")]])
paired.round(4)

### Accuracy by CRAG action

Correct keeps the refined textbook knowledge, Incorrect replaces it with web search results, and Ambiguous uses both.
Comparing the settings on the same claims shows where CRAG's correction helps or hurts.

In [ ]:
ACTIONS = ["correct", "ambiguous", "incorrect"]
by_action = (results.groupby("crag_action")
             .agg(claims=("id", "size"),
                  true_claims=("gold", lambda g: int((g == "true").sum())),
                  no_retrieval=("no_retrieval_correct", "mean"),
                  rag=("rag_correct", "mean"),
                  crag=("crag_correct", "mean"),
                  mean_max_score=("crag_max_score", "mean"),
                  empty_knowledge=("crag_knowledge_items", lambda n: int((n == 0).sum())),
                  mean_web_items=("crag_web_items", "mean"))
             .reindex([a for a in ACTIONS if a in set(results.crag_action)]))
by_action.round(3)

In [ ]:
ax = by_action[["no_retrieval", "rag", "crag"]].plot.bar(color=list(COLOURS.values()), figsize=(8, 4), rot=0)
ax.set_ylim(0, 1)
ax.set_ylabel("Accuracy")
ax.set_xlabel("CRAG action")
ax.set_title("Accuracy by CRAG action")
ax.set_xticklabels([f"{a}\n(n={n})" for a, n in zip(by_action.index, by_action.claims)])
ax.legend(list(SETTINGS.values()))
plt.tight_layout()
plt.savefig(RESULTS_DIR / f"fig_by_action_{TAG}.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# How much knowledge CRAG passed to the generator, and how long each setting took
knowledge_stats = pd.DataFrame({
    "claims": results.groupby("crag_action").size(),
    "mean items": results.groupby("crag_action").crag_knowledge_items.mean(),
    "mean words": results.groupby("crag_action").crag_knowledge_words.mean(),
    "claims with web results": results.groupby("crag_action").crag_web_items.apply(lambda s: int((s > 0).sum())),
    "claims with no knowledge": results.groupby("crag_action").crag_knowledge_items.apply(lambda s: int((s == 0).sum())),
}).reindex(by_action.index)
display(knowledge_stats.round(1))
print("Mean seconds per claim:", {name: round(results[f"seconds_{c}"].mean(), 1) for c, name in SETTINGS.items()})

### Comparison with the earlier runs in `results/` and the CRAG paper

`rag_pubhealth_n200_3b.csv` was made with `scripts/evaluate.py` and the same 3B model (no CRAG), and
`crag_pubhealth_n200_0.5b.csv` with the 0.5B model. The CRAG paper (Yan et al., 2024, Table 1) used LLaMA2 generators
and Wikipedia retrieval, so its numbers are a reference point rather than a direct comparison.

In [ ]:
rows_cmp = []
for path, model_name in [(RESULTS_DIR / "crag_pubhealth_n200_0.5b.csv", "Qwen2.5-0.5B (earlier run)"),
                         (RESULTS_DIR / "rag_pubhealth_n200_3b.csv", "Qwen2.5-3B (earlier run, no CRAG)")]:
    if path.exists() and path != RESULTS_PATH:
        old = as_labels(pd.read_csv(path), LABEL_COLUMNS)
        rows_cmp.append({"generator": model_name, "claims": len(old),
                         **{name: (old[col] == old.gold).mean() if col in old else np.nan
                            for col, name in SETTINGS.items()}})
rows_cmp.append({"generator": f"Qwen2.5-{MODEL_KEY.upper()} (this notebook)", "claims": len(results),
                 **summary.accuracy.to_dict()})
rows_cmp += [{"generator": "LLaMA2-hf-7b (CRAG paper)", "claims": np.nan,
              "No retrieval": np.nan, "Standard RAG": 0.489, "CRAG": 0.595},
             {"generator": "SelfRAG-LLaMA2-7b (CRAG paper)", "claims": np.nan,
              "No retrieval": np.nan, "Standard RAG": 0.390, "CRAG": 0.756}]
pd.DataFrame(rows_cmp).set_index("generator").round(3)

In [ ]:
# How closely this run reproduces the earlier 3B run on the settings both have (same claims, same model)
old_path = RESULTS_DIR / "rag_pubhealth_n200_3b.csv"
if MODEL_KEY == "3b" and old_path.exists():
    old = as_labels(pd.read_csv(old_path), LABEL_COLUMNS).drop_duplicates("claim")
    both = results.merge(old, on="claim", suffixes=("", "_earlier"))
    for col in ["no_retrieval", "rag"]:
        same = (both[col] == both[f"{col}_earlier"]).mean()
        print(f"{SETTINGS[col]}: same label as the earlier run for {same:.1%} of {len(both)} claims")

## 8. Ablations: results

In [ ]:
if ablations is not None:
    abl = results.merge(ablations, on="id")
    ALL = {**SETTINGS, **ABLATIONS}
    abl_summary = evaluate(abl, ALL)
    abl_summary["vs CRAG"] = abl_summary.accuracy - abl_summary.loc["CRAG", "accuracy"]
    display(abl_summary.round(3))
    display(pd.DataFrame([compare(abl, v, "crag", ALL) for v in ABLATIONS]).round(4))
    needs_web = abl.action_check != C
    print(f"Of the {needs_web.sum()} claims where CRAG searches the web, PubMed found results for "
          f"{int((abl.web_items[needs_web] > 0).sum())} with the rewritten query, "
          f"{int((abl.raw_web_items[needs_web] > 0).sum())} with the raw claim and "
          f"{int((abl.backoff_web_items[needs_web] > 0).sum())} with the back-off search")

In [ ]:
if ablations is not None:
    order = list(ALL.values())
    special = {"No retrieval": "#64748B", "Standard RAG": "#D97706", "CRAG": "#4F46E5", "Oracle context": "#10B981"}
    colours = [special.get(name, "#A5B4FC") for name in order]
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.barh(order[::-1], abl_summary.loc[order, "accuracy"][::-1], color=colours[::-1],
            xerr=[(abl_summary.accuracy - abl_summary.ci_low).loc[order][::-1],
                  (abl_summary.ci_high - abl_summary.accuracy).loc[order][::-1]], capsize=4)
    ax.axvline(always_true, ls="--", color="black", lw=1, label="always 'true'")
    ax.set_xlim(0, 1)
    ax.set_xlabel("Accuracy (95% bootstrap interval)")
    ax.set_title("Ablations: each variant changes one part of CRAG")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"fig_ablations_{TAG}.png", dpi=200, bbox_inches="tight")
    plt.show()

## 9. Threshold sensitivity

CRAG picks Correct when any passage scores at least `UPPER`, Incorrect when every passage scores below `LOWER`, and
Ambiguous otherwise. The forced-action variants give the prediction each action would lead to for every claim, so
CRAG's accuracy can be computed exactly for any pair of thresholds without running the generator again.

In [ ]:
if ablations is not None:
    scores_by_id = {i: json.loads(s) for i, s in zip(results.id, results.crag_scores)}
    pred = {C: dict(zip(abl.id, abl.all_correct)), I: dict(zip(abl.id, abl.all_incorrect)),
            A: dict(zip(abl.id, abl.all_ambiguous))}
    gold = dict(zip(abl.id, abl.gold))

    def accuracy_at(upper, lower):
        acts = {i: actions.decide_action(scores_by_id[i], upper=upper, lower=lower) for i in abl.id}
        return np.mean([pred[acts[i]][i] == gold[i] for i in abl.id]), acts

    uppers = np.round(np.arange(-0.5, 1.0, 0.1), 2)
    lowers = np.round(np.arange(-1.0, 0.5, 0.1), 2)
    grid = pd.DataFrame([[accuracy_at(u, l)[0] if l <= u else np.nan for u in uppers] for l in lowers],
                        index=lowers, columns=uppers)
    fig, ax = plt.subplots(figsize=(9, 6))
    im = ax.imshow(grid.values, origin="lower", cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(uppers)), uppers, rotation=45)
    ax.set_yticks(range(len(lowers)), lowers)
    ax.set_xlabel("UPPER threshold")
    ax.set_ylabel("LOWER threshold")
    u0, l0 = list(uppers).index(round(actions.UPPER, 2)), list(lowers).index(round(actions.LOWER, 2))
    ax.scatter([u0], [l0], marker="o", s=160, facecolors="none", edgecolors="red", lw=2, label="current thresholds")
    for y in range(len(lowers)):
        for x in range(len(uppers)):
            if not np.isnan(grid.values[y, x]):
                ax.text(x, y, f"{grid.values[y, x]:.2f}", ha="center", va="center", fontsize=6, color="white")
    fig.colorbar(im, label="CRAG accuracy")
    ax.set_title("CRAG accuracy for each pair of action thresholds")
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"fig_thresholds_{TAG}.png", dpi=200, bbox_inches="tight")
    plt.show()

    best = grid.stack().idxmax()
    acc_now, _ = accuracy_at(actions.UPPER, actions.LOWER)
    print(f"Current thresholds (UPPER {actions.UPPER}, LOWER {actions.LOWER}): accuracy {acc_now:.3f} "
          f"(main run: {results.crag_correct.mean():.3f})")
    print(f"Best on these claims: UPPER {best[1]}, LOWER {best[0]}: accuracy {grid.stack().max():.3f}. "
          "This is tuned on the test claims, so it is optimistic; use it to choose thresholds on held-out data.")

## 10. Quality of the knowledge given to the generator

Accuracy only shows the final label. These checks look at the knowledge itself, without the generator:

1. **Evidence table**: every sentence that RAG or CRAG passed to the generator, scored against the claim by the
   retrieval evaluator (`question, sentence_idx, sentence, relevance_score, relevant`). CRAG selects strips with this
   same evaluator, so its scores favour CRAG by design.
2. **Closeness to the fact-checkers' explanation**: Contriever cosine similarity between each knowledge item and the
   PubHealth explanation for the claim, an independent check of whether the right evidence was found.
3. **Separation test**: does the evaluator (and Contriever) score a claim's own explanation above another claim's
   explanation? (`question, score_gold_context, score_wrong_context, correctly_separated`)

In [ ]:
@torch.no_grad()
def t5_scores(question, texts, batch_size=32):
    """Batched crag.evaluator.evaluate_relevance (same model, input format and truncation)."""
    out = []
    for i in range(0, len(texts), batch_size):
        batch = [f"question: {question} document: {t}" for t in texts[i:i + batch_size]]
        enc = eval_tokenizer(batch, truncation=True, max_length=crag_evaluator.MAX_LENGTH, padding=True,
                             return_tensors="pt")
        scores = eval_model(enc["input_ids"].to(crag_evaluator.DEVICE), enc["attention_mask"].to(crag_evaluator.DEVICE))
        out += scores.reshape(-1).tolist()
    return out


# Check the batched scores against the group's function on one claim
sample_claim = results.claim.iloc[0]
sample = retrieve(index, sample_claim, k=3)
gap = max(abs(a - score_fn(sample_claim, t)) for a, t in zip(t5_scores(sample_claim, sample), sample))
print(f"Largest difference from evaluate_relevance: {gap:.2e}")

In [ ]:
def sentences(texts, min_words=4):
    out = []
    for text in texts:
        for s in split_into_strips(text, sentences_per_strip=1):
            if len(s.split()) >= min_words and s not in out:
                out.append(s)
    return out


evidence_rows = []
if knowledge:
    for rec in tqdm(knowledge.values(), desc="Evidence table"):
        for method, texts in (("rag", rec["passages"]), ("crag", crag_knowledge_of(rec))):
            sents = sentences(texts)
            for idx, (s, score) in enumerate(zip(sents, t5_scores(rec["claim"], sents))):
                evidence_rows.append({"id": rec["id"], "method": method, "question": rec["claim"],
                                      "sentence_idx": idx, "sentence": s, "relevance_score": round(score, 4),
                                      "relevant": score >= 0})
    evidence = pd.DataFrame(evidence_rows)
    evidence.to_csv(RESULTS_DIR / f"evidence_{TAG}.csv", index=False)
    per_claim = evidence.groupby(["method", "id"]).agg(sentences=("sentence", "size"),
                                                      mean_score=("relevance_score", "mean"),
                                                      share_relevant=("relevant", "mean")).reset_index()
    display(per_claim.groupby("method")[["sentences", "mean_score", "share_relevant"]].mean().round(3)
            .rename(index=SETTINGS))
    display(evidence[evidence.method == "crag"].head(10)[["question", "sentence_idx", "sentence",
                                                          "relevance_score", "relevant"]])

In [ ]:
@torch.no_grad()
def embed(texts):
    vecs = _encode(index.model, index.tokenizer, list(texts))
    return torch.nn.functional.normalize(vecs.float(), dim=-1)


gold_sim_rows = []
if knowledge:
    for rec in tqdm(knowledge.values(), desc="Closeness to the explanation"):
        expl = embed([explanations[rec["id"]]])[0]
        row = {"id": rec["id"], "action": rec["action"]}
        for method, texts in (("rag", rec["passages"]), ("crag", crag_knowledge_of(rec))):
            texts = [t for t in texts if t and t.strip()]
            row[method] = float((embed(texts) @ expl).max()) if texts else np.nan
        gold_sim_rows.append(row)
    gold_sim = pd.DataFrame(gold_sim_rows)
    print("Best cosine similarity between any knowledge item and the fact-checkers' explanation "
          "(NaN = no knowledge given):")
    display(pd.concat([gold_sim[["rag", "crag"]].mean().rename("all claims"),
                       gold_sim.groupby("action")[["rag", "crag"]].mean().T], axis=1).round(3)
            .rename(index=SETTINGS))
    print(f"CRAG gave no knowledge for {gold_sim.crag.isna().sum()} claims; RAG always gives {K} passages.")

In [ ]:
# Separation test: gold = the claim's own explanation, wrong = the next claim's explanation
sep = claims[["claim", "explanation"]].copy()
sep["wrong"] = np.roll(sep.explanation.to_numpy(), -1)
sep = sep[sep.explanation.str.strip().astype(bool) & (sep.explanation != sep.wrong)]
rows_t5, rows_con = [], []
for r in tqdm(sep.itertuples(), total=len(sep), desc="Separation test"):
    g, w = t5_scores(r.claim, [r.explanation, r.wrong])
    rows_t5.append({"question": r.claim, "score_gold_context": g, "score_wrong_context": w,
                    "correctly_separated": g > w})
    q = _encode(index.model, index.tokenizer, [r.claim, r.explanation, r.wrong])
    rows_con.append({"question": r.claim, "score_gold_context": float(q[1] @ q[0]),
                     "score_wrong_context": float(q[2] @ q[0]), "correctly_separated": float(q[1] @ q[0]) > float(q[2] @ q[0])})
separation = {"T5 retrieval evaluator (CRAG)": pd.DataFrame(rows_t5),
              "Contriever similarity (standard RAG)": pd.DataFrame(rows_con)}
for name, table in separation.items():
    table.to_csv(RESULTS_DIR / f"separation_{TAG.replace('_' + MODEL_KEY, '')}_{'t5' if 'T5' in name else 'contriever'}.csv",
                 index=False)
display(pd.DataFrame({name: {"pairs": len(t), "correctly separated": t.correctly_separated.mean(),
                             "mean gold score": t.score_gold_context.mean(),
                             "mean wrong score": t.score_wrong_context.mean()}
                      for name, t in separation.items()}).round(3))
separation["T5 retrieval evaluator (CRAG)"].head()

## 11. Example responses

The accuracy runs skip the explanation. Here a few claims are answered again with `explain=True`, which adds the
evidence sentences, a short reasoning and the response a user would see. The examples show a claim that CRAG fixed,
one it got wrong, and one that both RAG and CRAG got right.

In [ ]:
picks = []
for mask in [results.crag_correct & ~results.rag_correct,
             ~results.crag_correct & results.rag_correct,
             results.crag_correct & results.rag_correct]:
    if mask.any():
        picks.append(results[mask].iloc[0])

for row in picks[:N_EXAMPLES]:
    print("#" * 100)
    print(f"CLAIM: {row.claim}\nGOLD:  {row.gold}   |   RAG: {row.rag}   |   CRAG: {row.crag} ({row.crag_action})")
    rag_out = run_rag(row.claim, index, model, tokenizer, k=K)
    knowledge_ex, action, _ = correct_knowledge(row.claim, retrieve(index, row.claim, k=K), eval_model,
                                                eval_tokenizer, gen_model=model, gen_tokenizer=tokenizer)
    crag_out = predict_label(model, tokenizer, row.claim, knowledge_ex)
    print(f"\n--- Standard RAG response (P(true) = {rag_out.probs['true']:.2f}) ---\n{rag_out.response}")
    print(f"\n--- CRAG response, action {action} (P(true) = {crag_out.probs['true']:.2f}) ---\n{crag_out.response}\n")

## 12. Save the summary

In [ ]:
summary_path = RESULTS_DIR / f"summary_{TAG}.json"
out = {
    "generator": MODELS[MODEL_KEY]["model_name"],
    "device": DEVICE,
    "dtype": str(next(model.parameters()).dtype),
    "claims": len(results),
    "k": K,
    "thresholds": {"upper": actions.UPPER, "lower": actions.LOWER},
    "web_search": "Tavily + PubMed" if os.getenv("TAVILY_API_KEY") else "PubMed only",
    "always_true_accuracy": always_true,
    "main": summary.round(4).to_dict(orient="index"),
    "paired": paired.round(4).to_dict(orient="records"),
    "by_crag_action": by_action.round(4).to_dict(orient="index"),
}
if ablations is not None:
    out["ablations"] = abl_summary.round(4).to_dict(orient="index")
    out["threshold_best"] = {"upper": float(best[1]), "lower": float(best[0]), "accuracy": float(grid.stack().max())}
if knowledge:
    out["evidence"] = per_claim.groupby("method")[["sentences", "mean_score", "share_relevant"]].mean().round(4).to_dict(orient="index")
    out["closeness_to_explanation"] = gold_sim[["rag", "crag"]].mean().round(4).to_dict()
out["separation"] = {name: {"correctly_separated": float(t.correctly_separated.mean()),
                            "mean_gold": float(t.score_gold_context.mean()),
                            "mean_wrong": float(t.score_wrong_context.mean())} for name, t in separation.items()}
summary_path.write_text(json.dumps(out, indent=2, default=float))
print("Saved", summary_path.relative_to(ROOT))